In [0]:
from gold_transforms import (
    build_dim_date, build_dim_geography, build_fact_orders, build_fact_order_lines
)

dbutils.widgets.text("env", "dev")
env     = dbutils.widgets.get("env")
SV, GD  = f"{env}_silver", f"{env}_gold"
G       = f"{GD}.common"
regions = ["africa", "america", "asia", "europe", "middle_east"]

def load(target, select_sql):
    spark.sql(f"INSERT OVERWRITE {target} {select_sql}")
    print(f"✔ {target}: {spark.table(target).count():,} rows")

def load_df(target, df):
    """INSERT OVERWRITE from a DataFrame."""
    view = "_stg_" + target.split(".")[-1]
    df.createOrReplaceTempView(view)
    spark.sql(f"INSERT OVERWRITE {target} SELECT * FROM {view}")
    print(f"✔ {target}: {spark.table(target).count():,} rows")


In [0]:
load_df(f"{G}.dim_date", build_dim_date(spark, "1992-01-01", "1999-12-31"))
load_df(f"{G}.dim_geography", build_dim_geography(spark, f"{SV}.common.nations", f"{SV}.common.regions"))
load(f"{G}.dim_product", f"""
  SELECT product_id, product_name, manufacturer, brand, product_type, size, container, retail_price
  FROM {SV}.common.products""")
load(f"{G}.dim_supplier", f"""
  SELECT supplier_id, supplier_name, nation_id, phone, account_balance
  FROM {SV}.common.suppliers""")


In [0]:
for r in regions:
    load(f"{GD}.{r}.dim_customer", f"""
      SELECT customer_id, customer_name, market_segment, nation_id, phone, account_balance
      FROM {SV}.{r}.customers""")
    load_df(f"{GD}.{r}.fact_orders",
            build_fact_orders(spark, f"{SV}.{r}.orders", f"{SV}.{r}.order_lines"))
    load_df(f"{GD}.{r}.fact_order_lines",
            build_fact_order_lines(spark, f"{SV}.{r}.order_lines"))


In [0]:
%sql
-- 1. Row counts by region (from the combined views)
SELECT sales_region, COUNT(*) AS order_lines, ROUND(SUM(net_amount), 0) AS net_revenue
FROM dev_gold.common.vw_order_lines_all
GROUP BY sales_region ORDER BY sales_region;



In [0]:
%sql
-- 2. Orphan check: every value should be 0 (foreign keys are not enforced, so we verify them)
SELECT
  (SELECT COUNT(*) FROM dev_gold.common.vw_order_lines_all f
     LEFT ANTI JOIN dev_gold.common.dim_product p ON f.product_id = p.product_id)   AS missing_products,
  (SELECT COUNT(*) FROM dev_gold.common.vw_order_lines_all f
     LEFT ANTI JOIN dev_gold.common.dim_supplier s ON f.supplier_id = s.supplier_id) AS missing_suppliers,
  (SELECT COUNT(*) FROM dev_gold.common.vw_orders_all o
     LEFT ANTI JOIN dev_gold.common.dim_date d ON o.order_date_key = d.date_key)     AS missing_dates;


In [0]:
%sql
-- 3. A real business question (a preview of what Genie will answer later)
SELECT d.year, f.sales_region, ROUND(SUM(f.net_amount) / 1e6, 2) AS net_revenue_musd
FROM dev_gold.common.vw_order_lines_all f
JOIN dev_gold.common.dim_date d ON f.ship_date_key = d.date_key
GROUP BY ALL ORDER BY d.year, f.sales_region;